# Phase C — UNet Path-Loss Surrogate Training
Implements §5 of the build spec: 8-channel input, UNet base-64, MSE loss, seeds {0,1,2},
baselines (FSPL / fitted log-distance / 3GPP 38.901 InH with exact LOS), RMSE_dB/MAE/bias,
side-by-side figure, ONNX export with parity check.

**Runtime → GPU (T4 or better).** Nothing runs on your own computer.

**Dataset**: generated locally by `SIM/phase_b_dataset.py` (~1.5 GB, not in git).
Upload the `SIM/dataset/` folder to your Google Drive as `indoor-walk-test-dataset/`
(or let cell 3 regenerate it here — several hours on Colab's CPU).

**Rules enforced here**: R1 (one output), R4 (splits by position, from committed
splits.json), R5 (test touched once, in the final evaluation section only),
R6 (all constants read from manifest.json), R9 (multi-seed mean ± std).

In [ ]:
import os
REPO = 'github.com/cgm2179/indoor-walk-test.git'
if not os.path.exists('indoor-walk-test'):
    if os.system(f'git clone --depth 1 https://{REPO}') != 0:
        from getpass import getpass
        token = getpass('GitHub fine-grained token: ')
        assert os.system(f'git clone --depth 1 https://{token}@{REPO}') == 0, 'clone failed'
%cd indoor-walk-test
import torch, json, numpy as np
print('GPU:', torch.cuda.is_available())
manifest = json.load(open('SIM/manifest.json'))
NORM = manifest['norm']; PL_LO, PL_RNG = NORM['pl_min_db'], NORM['pl_range_db']
H, W = manifest['grid_shape']; SIGMA = manifest['tx_blob_sigma_cells']
print('grid', (H, W), '| clip', PL_LO, '-', PL_LO + PL_RNG, 'dB')

In [ ]:
from pathlib import Path
DATASET = Path('SIM/dataset')
# splits.json IS in git but the 20 shard_*.npz are gitignored -- so test for
# shards, not for the folder, then fall back to Drive.
if not any(DATASET.glob('shard_*.npz')):
    from google.colab import drive
    drive.mount('/content/drive')
    cand = Path('/content/drive/MyDrive/indoor-walk-test-dataset')
    assert any(cand.glob('shard_*.npz')), (
        'No shards found. On your Mac, upload the whole SIM/dataset folder '
        '(1.2 GB: splits.json + 20 shard_*.npz + dataset_meta.json) to '
        'Google Drive and name the folder exactly indoor-walk-test-dataset. '
        'Alternative: regenerate here with  !python SIM/phase_b_dataset.py  '
        '(several hours on Colab CPU).')
    DATASET = cand
print('dataset from:', DATASET)

splits = json.load(open(DATASET / 'splits.json'))
shards = sorted(DATASET.glob('shard_*.npz'))
print(f'{len(shards)} shards')
tx_all, ff_all, tg_all, pid_all = [], [], [], []
for s in shards:
    d = np.load(s)
    tx_all.append(d['tx_pos']); ff_all.append(d['freq_feat'])
    tg_all.append(d['target']); pid_all.append(d['pos_id'])
TX = np.concatenate(tx_all); FF = np.concatenate(ff_all)
TG = np.concatenate(tg_all)                      # float16 (N, H, W)
PID = np.concatenate(pid_all)
print(f'{len(TG)} samples, targets {TG.shape} {TG.dtype}, '
      f'{TG.nbytes/1e9:.2f} GB in RAM')
idx = {k: np.nonzero(np.isin(PID, splits[k]))[0] for k in ('train','val','test')}
print({k: len(v) for k, v in idx.items()})

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

grid = np.load('SIM/grid_model.npy')
inside = torch.from_numpy(np.load('SIM/inside_mask.npy'))
onehot = np.stack([(grid == c) for c in range(6)]).astype(np.float32)  # (6,H,W)
yy, xx = np.mgrid[0:H, 0:W].astype(np.float32)

def tx_blob(x, y):
    return np.exp(-((xx - x)**2 + (yy - y)**2) / (2 * SIGMA**2)).astype(np.float32)

class PLData(Dataset):
    def __init__(self, ids): self.ids = ids
    def __len__(self): return len(self.ids)
    def __getitem__(self, i):
        j = self.ids[i]
        x = np.empty((8, H, W), np.float32)
        x[:6] = onehot
        x[6] = tx_blob(*TX[j])
        x[7] = FF[j]
        return torch.from_numpy(x), torch.from_numpy(TG[j].astype(np.float32))

def loaders(batch):
    return (DataLoader(PLData(idx['train']), batch_size=batch, shuffle=True,
                       num_workers=2, pin_memory=True, drop_last=True),
            DataLoader(PLData(idx['val']), batch_size=batch, num_workers=2))

In [ ]:
import torch.nn as nn

def block(i, o):
    return nn.Sequential(nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(True),
                         nn.Conv2d(o, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(True))

class UNet(nn.Module):
    """4 down / 4 up, base width configurable (spec C.2)."""
    def __init__(self, base=64, in_ch=8):
        super().__init__()
        ch = [base, base*2, base*4, base*8]
        self.enc = nn.ModuleList(); prev = in_ch
        for c in ch: self.enc.append(block(prev, c)); prev = c
        self.pool = nn.MaxPool2d(2)
        self.mid = block(prev, prev*2)
        self.up, self.dec = nn.ModuleList(), nn.ModuleList()
        prev = prev*2
        for c in reversed(ch):
            self.up.append(nn.ConvTranspose2d(prev, c, 2, 2))
            self.dec.append(block(2*c, c)); prev = c
        self.head = nn.Conv2d(prev, 1, 1)          # R1: one output channel
    def forward(self, x):
        skips = []
        for e in self.enc:
            x = e(x); skips.append(x); x = self.pool(x)
        x = self.mid(x)
        for u, d in zip(self.up, self.dec):
            x = d(torch.cat([u(x), skips.pop()], 1))
        return self.head(x)[:, 0]

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'{sum(p.numel() for p in UNet().parameters())/1e6:.1f} M params')

In [ ]:
import time, math

GRAD_W = 0.1     # spec C.3: optional gradient-matching term

def grad_l1(a, b):
    return ((a[:,1:,:]-a[:,:-1,:]) - (b[:,1:,:]-b[:,:-1,:])).abs().mean() + \
           ((a[:,:,1:]-a[:,:,:-1]) - (b[:,:,1:]-b[:,:,:-1])).abs().mean()

def train_one(seed, base=64, lr=1e-3, batch=16, epochs=150, patience=15):
    torch.manual_seed(seed); np.random.seed(seed)
    net = UNet(base).to(dev)
    opt = torch.optim.Adam(net.parameters(), lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs, eta_min=1e-5)
    scaler = torch.amp.GradScaler(dev)
    tr_dl, va_dl = loaders(batch)
    best, best_ep, hist = 1e9, -1, []
    for ep in range(epochs):
        net.train(); t0 = time.time()
        for x, y in tr_dl:
            x, y = x.to(dev, non_blocking=True), y.to(dev, non_blocking=True)
            with torch.amp.autocast(dev):
                p = net(x)
                loss = torch.nn.functional.mse_loss(p, y) + GRAD_W * grad_l1(p, y)
            opt.zero_grad(); scaler.scale(loss).backward()
            scaler.step(opt); scaler.update()
        sched.step()
        net.eval(); se = n = 0
        with torch.no_grad():
            for x, y in va_dl:
                p = net(x.to(dev)).cpu()
                se += ((p - y)**2).sum().item(); n += y.numel()
        rmse_db = PL_RNG * math.sqrt(se / n)
        hist.append(rmse_db)
        if rmse_db < best:
            best, best_ep = rmse_db, ep
            torch.save(net.state_dict(), f'unet_seed{seed}_best.pt')
        print(f'seed {seed} ep {ep+1:3d}  val RMSE {rmse_db:5.2f} dB  '
              f'best {best:5.2f}  ({time.time()-t0:.0f}s)')
        if ep - best_ep >= patience:
            print(f'early stop (patience {patience})'); break
    return best

In [ ]:
# R9: seeds {0,1,2}. Set QUICK=True for a single-seed smoke run first.
QUICK = True
SEEDS = [0] if QUICK else [0, 1, 2]
results = {s: train_one(s) for s in SEEDS}
vals = list(results.values())
print(f'val RMSE over {len(SEEDS)} seed(s): '
      f'{np.mean(vals):.2f} ± {np.std(vals):.2f} dB  (target <= 3 dB)')

## Baselines (spec C.4) — evaluated on **validation** for selection
(a) FSPL-only · (b) log-distance with n fitted on train · (c) 3GPP TR 38.901
InH-Office dual-state with the LOS indicator computed **exactly** by the Phase A
ray-march (zero wall crossings ⇒ LOS). Test-set numbers appear only in the final
section (R5).

In [ ]:
import importlib.util
spec_pa = importlib.util.spec_from_file_location('pa', 'SIM/phase_a.py')
pa = importlib.util.module_from_spec(spec_pa); spec_pa.loader.exec_module(pa)
CELL = manifest['cell_size_m']
F_LO, F_HI = np.log10(NORM['freq_log_lo_mhz']), np.log10(NORM['freq_log_hi_mhz'])
dist_m = lambda j: np.maximum(np.hypot(xx - TX[j,0], yy - TX[j,1]) * CELL, 1.0)
f_of = lambda j: 10 ** (FF[j] * (F_HI - F_LO) + F_LO)
ins = inside.numpy()

def fspl_map(j):
    return 32.44 + 20*np.log10(f_of(j)) - 60 + 20*np.log10(dist_m(j))

# (b) fit n on a subsample of train (same calibration budget as the UNet gets)
sub = np.random.default_rng(0).choice(idx['train'], 400, replace=False)
num = den = 0.0
for j in sub:
    pl = TG[j].astype(np.float32) * PL_RNG + PL_LO
    ex = pl - (32.44 + 20*np.log10(f_of(j)) - 60)
    ld = 10*np.log10(dist_m(j))
    num += (ex[ins] * ld[ins]).sum(); den += (ld[ins]**2).sum()
N_FIT = num / den
print(f'fitted log-distance n = {N_FIT:.2f}')

def logdist_map(j):
    return 32.44 + 20*np.log10(f_of(j)) - 60 + 10*N_FIT*np.log10(dist_m(j))

def inh38901_map(j, los):
    d = dist_m(j); fg = np.log10(f_of(j)/1000)*20
    pl_los = 32.4 + 17.3*np.log10(d) + fg
    pl_nlos = np.maximum(pl_los, 17.3 + 38.3*np.log10(d) + 24.9*np.log10(f_of(j)/1000))
    return np.where(los, pl_los, pl_nlos)

def eval_maps(ids, model=None, kind='unet', los_cache=None):
    se = ae = be = n = 0
    for j in ids:
        gt = TG[j].astype(np.float32) * PL_RNG + PL_LO
        if kind == 'unet':
            x = np.empty((8, H, W), np.float32); x[:6] = onehot
            x[6] = tx_blob(*TX[j]); x[7] = FF[j]
            with torch.no_grad():
                p = model(torch.from_numpy(x)[None].to(dev))[0].cpu().numpy()
            pr = np.clip(p, 0, 1) * PL_RNG + PL_LO
        elif kind == 'fspl':    pr = np.clip(fspl_map(j), PL_LO, PL_LO+PL_RNG)
        elif kind == 'logdist': pr = np.clip(logdist_map(j), PL_LO, PL_LO+PL_RNG)
        elif kind == '38901':
            key = tuple(TX[j])
            if key not in los_cache:
                _, k = pa.pathloss_map(np.load('SIM/grid_model.npy'),
                                       (float(TX[j,0]), float(TX[j,1])),
                                       3500.0, CELL, return_crossings=True)
                los_cache[key] = k == 0
            pr = np.clip(inh38901_map(j, los_cache[key]), PL_LO, PL_LO+PL_RNG)
        e = (pr - gt)[ins]
        se += (e**2).sum(); ae += np.abs(e).sum(); be += e.sum(); n += e.size
    return dict(rmse=np.sqrt(se/n), mae=ae/n, bias=be/n)

net = UNet().to(dev); net.load_state_dict(torch.load(f'unet_seed{SEEDS[0]}_best.pt')); net.eval()
val_sub = np.random.default_rng(1).choice(idx['val'], 200, replace=False)
los_cache = {}
for kind in ('fspl', 'logdist', '38901', 'unet'):
    r = eval_maps(val_sub, net, kind, los_cache)
    print(f"VAL {kind:8s} RMSE {r['rmse']:6.2f}  MAE {r['mae']:6.2f}  bias {r['bias']:+6.2f} dB")

## Final test evaluation — R5: this section runs ONCE, after all selection is done.

In [ ]:
test_sub = idx['test']
for kind in ('fspl', 'logdist', '38901', 'unet'):
    r = eval_maps(test_sub, net, kind, los_cache)
    print(f"TEST {kind:8s} RMSE {r['rmse']:6.2f}  MAE {r['mae']:6.2f}  bias {r['bias']:+6.2f} dB")

In [ ]:
import matplotlib.pyplot as plt
js = idx['test'][:3]
fig, axes = plt.subplots(3, 3, figsize=(16, 7.5), squeeze=False)
for r, j in enumerate(js):
    gt = TG[j].astype(np.float32) * PL_RNG + PL_LO
    x = np.empty((8, H, W), np.float32); x[:6] = onehot
    x[6] = tx_blob(*TX[j]); x[7] = FF[j]
    with torch.no_grad():
        p = net(torch.from_numpy(x)[None].to(dev))[0].cpu().numpy()
    pr = np.clip(p, 0, 1) * PL_RNG + PL_LO
    for c, (img, ttl, cm, lo, hi) in enumerate([
            (gt, 'simulated PL (dB)', 'turbo', 40, 170),
            (pr, 'UNet prediction', 'turbo', 40, 170),
            (np.abs(pr-gt), '|error| (dB)', 'magma', 0, 10)]):
        a = axes[r][c]
        im = a.imshow(np.where(ins, img, np.nan), cmap=cm, vmin=lo, vmax=hi)
        a.set_title(ttl if r == 0 else '', fontsize=10); a.axis('off')
        fig.colorbar(im, ax=a, shrink=0.7)
plt.tight_layout()

In [ ]:
# D.3 physical sanity checks on the trained model
j = idx['test'][0]
x = np.empty((8, H, W), np.float32); x[:6] = onehot
x[6] = tx_blob(*TX[j]); fx, fy = TX[j]
x[7] = FF[j]
with torch.no_grad():
    p = net(torch.from_numpy(x)[None].to(dev))[0].cpu().numpy() * PL_RNG + PL_LO
ray = p[int(fy), int(fx):int(fx)+40]
print('monotone LOS decay (should mostly increase):',
      (np.diff(ray) > -0.5).mean().round(2))
fspl_here = fspl_map(j)
print('cells below FSPL:', f'{100*(p < fspl_here - 1)[ins].mean():.2f}% (want ~0)')
x24 = x.copy(); x24[7] = 0.0   # 2.4 GHz feature
with torch.no_grad():
    p24 = net(torch.from_numpy(x24)[None].to(dev))[0].cpu().numpy() * PL_RNG + PL_LO
print('median(PL@2.4GHz <= PL@current):', f'{np.median((p24 <= p + 0.5)[ins]):.0f} (want 1)')

In [ ]:
# F.1/F.2: export + parity check (max |ONNX - PyTorch| <= 0.1 dB on 50 test maps)
net.cpu().eval()
torch.onnx.export(net, torch.zeros(1, 8, H, W), 'pl_unet.onnx',
                  input_names=['x'], output_names=['pl_norm'], opset_version=17)
import onnxruntime as ort_rt
sess = ort_rt.InferenceSession('pl_unet.onnx')
worst = 0
for j in idx['test'][:50]:
    x = np.empty((8, H, W), np.float32); x[:6] = onehot
    x[6] = tx_blob(*TX[j]); x[7] = FF[j]
    with torch.no_grad():
        pt = net(torch.from_numpy(x)[None])[0].numpy()
    ox = sess.run(None, {'x': x[None]})[0][0]
    worst = max(worst, np.abs(pt - ox).max() * PL_RNG)
print(f'ONNX parity: max |diff| {worst:.4f} dB (must be <= 0.1)')
assert worst <= 0.1, 'PARITY FAIL - do not ship this file'
net.to(dev)
from google.colab import files
files.download('pl_unet.onnx')   # place at SIM/web/pl_unet.onnx in the repo